# CNN Window-Sequence Accuracy

Compare the current per-slice CNN baseline against a flattened linear window head and uni-/bi-directional LSTMs (1 and 2 layers). The target remains the CADS kidney/liver **organ proxy**, not true tumor masks.

The window length is computed once as the training-only 95th percentile of target axial extent. A window is positive if any constituent slice contains target.

In [ ]:
!nvidia-smi
!pip -q install nibabel huggingface_hub datasets pyyaml tqdm scikit-learn psutil matplotlib

from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/Colab Notebooks/CNN_window_sequence_accuracy'
os.chdir(PROJECT_ROOT)
DATA_ROOT = '/content/data/cads'  # local SSD: never train from Drive FUSE
os.makedirs(DATA_ROOT, exist_ok=True)
print('project:', os.getcwd(), '| data:', DATA_ROOT)

## 1. Download and inspect the cohort

Only part 551 is needed: labels 2/3 are kidneys for KiTS and label 5 is liver for LiTS. Download bulk NIfTI data to local Colab SSD, while code and checkpoints remain on Drive. The default experiment uses 300 paired volumes and a subset-stratified 70/15/15 split (about 210 train, 45 validation, and 45 independent test volumes).

In [ ]:
from huggingface_hub import login
from pathlib import Path
n_images = len(list(Path(DATA_ROOT).rglob('images/*.nii*')))
if n_images < 300:
    print(f'Found {n_images}/300 images; downloading the missing cohort files...')
    login()
    !python scripts/download_subset.py --subsets 0003_kits21 0004_lits --local-dir {DATA_ROOT} --max-files 600 --parts 551

!python scripts/analyze_data.py --config configs/config.yaml --data-root {DATA_ROOT} --out results

## 2. Train the current per-slice CNN baseline

The ResNet-18 baseline is trained end-to-end once. Its encoder is then frozen and reused by every temporal head, isolating the value of sequence aggregation from differences in image features.

In [ ]:
%%time
!python scripts/train_baseline.py --config configs/config.yaml --data-root {DATA_ROOT}

## 3. Compute q95 window size and cache embeddings

Axial extents are measured only from training masks on the resized 64-slice grid. `W = ceil(q95(extent))`; validation and test masks cannot influence W. Each valid stride-1 window is positive if any of its W slices is positive.

In [ ]:
!python scripts/prepare_windows.py --config configs/config.yaml --data-root {DATA_ROOT}

import json
spec = json.load(open('results/window_q95.json'))
print('window size:', spec['window_size'], 'slices')
print('train extents: min/median/max =', spec['extent_min'], spec['extent_median'], spec['extent_max'])

## 4. Train temporal heads

Models: flattened-window linear classifier; uni-LSTM with 1/2 layers; bi-LSTM with 1/2 layers. All consume the same frozen W×F embeddings, train on the same balanced window sampler, and select checkpoints using validation AP.

In [ ]:
%%time
!python scripts/train_temporal.py --config configs/config.yaml

## 5. Held-out comparison

Thresholds are chosen on validation windows only, then frozen for test. Report raw window accuracy plus equal-weight macro volume accuracy, balanced accuracy, F1, AP, AUROC, per-subset metrics, volume-grouped confidence intervals, parameters, and temporal-head latency. Because overlapping windows are correlated, the effective test sample size is the number of test volumes—not the much larger window count—and confidence intervals resample whole volumes.

In [ ]:
!python scripts/evaluate_models.py --config configs/config.yaml --out results

import pandas as pd
from IPython.display import Image, display
from pathlib import Path
summary = json.load(open('results/window_comparison.json'))
print('independent test volumes:', summary['n_test_volumes'])
print('correlated test windows:', summary['n_test_windows'])
print('test volumes by subset:', summary['test_volumes_by_subset'])
display(pd.read_csv('results/window_comparison.csv'))
for name in ['accuracy_comparison.png', 'roc_pr_curves.png', 'accuracy_vs_latency.png']:
    display(Image(str(Path('results') / name)))